In [6]:
import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory


In [7]:
peso_max = 60

itens = [1,2,3,4,5]

pesos = [52,23,27,15,7]
valores = [100,60,70,15,15]



In [8]:
dict(zip(itens, valores))

{1: 100, 2: 60, 3: 70, 4: 15, 5: 15}

In [11]:
model = pyo.ConcreteModel()

model.itens = pyo.Set(initialize=itens)
model.pesos = pyo.Param(model.itens, initialize=dict(zip(itens, pesos)))
model.valores = pyo.Param(model.itens, initialize=dict(zip(itens, valores)))
model.x = pyo.Var(model.itens, domain=Binary)



In [12]:
# Func objetivo

def objetivo(model):
    return sum(model.valores[i]*model.x[i] for i in model.itens)

model.obj = pyo.Objective(rule=objetivo, sense=pyo.maximize)

In [14]:
# restricoes

def restricao_peso(model):
    return sum(model.pesos[i]*model.x[i] for i in model.itens) <= peso_max

model.restricao_peso = pyo.Constraint(rule=restricao_peso)

In [15]:
model.pprint()

1 Set Declarations
    itens : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    5 : {1, 2, 3, 4, 5}

2 Param Declarations
    pesos : Size=5, Index=itens, Domain=Any, Default=None, Mutable=False
        Key : Value
          1 :    52
          2 :    23
          3 :    27
          4 :    15
          5 :     7
    valores : Size=5, Index=itens, Domain=Any, Default=None, Mutable=False
        Key : Value
          1 :   100
          2 :    60
          3 :    70
          4 :    15
          5 :    15

1 Var Declarations
    x : Size=5, Index=itens
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :     0 :  None :     1 : False :  True : Binary
          2 :     0 :  None :     1 : False :  True : Binary
          3 :     0 :  None :     1 : False :  True : Binary
          4 :     0 :  None :     1 : False :  True : Binary
          5 :     0 :  None :     1 : False :  True : Binary

1 Ob

In [16]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\joaon\AppData\Local\Temp\tmp2479y7p_.cplex.log' open.
CPLEX> Problem 'C:\Users\joaon\AppData\Local\Temp\tmpcas9n9vs.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\joaon\AppData\Local\Temp\tmpcas9n9vs.pyomo.lp
Objective sense      : Maximize
Variables            :       5  [Binary: 5]
Objective nonzeros   :       5
Linear constraints   :       1  [Less: 1]
  Nonzeros           :       5
  RHS nonzeros       :       1

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objective nonzeros   : Min   : 15.00000         Max   :

In [18]:
for i in model.itens:
    print(f'Item {i}: { "Selecionado" if model.x[i].value == 1 else "Não selecionado" }')

Item 1: Não selecionado
Item 2: Selecionado
Item 3: Selecionado
Item 4: Não selecionado
Item 5: Selecionado
